# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nLicense: {metadata.license}\nPublished: {metadata.datePublished}\nKeywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all `RecordSet` and for each, inspect available fields and their columns (referencing only via their `@id`).

In [ ]:
# List all RecordSet IDs
if hasattr(metadata, "record_set") and metadata.record_set:
    record_sets = [r["@id"] if isinstance(r, dict) else r for r in metadata.record_set]
else:
    # Fallback: Try to parse recordsets from metadata (older schemas)
    record_sets = [rs["@id"] for rs in getattr(metadata, "_json", {}).get("recordSet", [])]
    if not record_sets:
        # Try to load from fields typical in Croissant schemas
        if hasattr(metadata, "_json"):
            for k, v in metadata._json.items():
                if isinstance(v, dict) and v.get("@type", "").endswith("RecordSet"):
                    record_sets.append(v["@id"])

print(f"Available Record Sets (`@id`): {record_sets}")

for rs_id in record_sets:
    try:
        rs = dataset.md.find(rs_id)
        print(f"\nRecord Set: {rs_id}")
        fields = getattr(rs, "field", [])
        if isinstance(fields, dict) or isinstance(fields, str):
            fields = [fields]
        print("Fields:`@id`s:")
        for field in fields:
            # field is often a dict with '@id' or just an '@id' string
            f_id = field["@id"] if isinstance(field, dict) and "@id" in field else (field if isinstance(field, str) else None)
            print(f" - {f_id}")
    except Exception as e:
        print(f"Failed to enumerate {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We select the main `RecordSet` ID. If unsure which to choose, pick the first available/printed above.

In [ ]:
# For demonstration, use the first record set found
if record_sets:
    main_record_set_id = record_sets[0]
else:
    raise RuntimeError("No record sets found in this dataset.")
print(f"Selected main RecordSet for analysis: {main_record_set_id}")

# Extract records for all available record sets
dataframes = {}
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded record set {record_set}: shape = {dataframes[record_set].shape}")
    except Exception as e:
        print(f"Failed to load records for {record_set}: {e}")

print("Columns in main record set:")
if main_record_set_id in dataframes and not dataframes[main_record_set_id].empty:
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No data found for record set {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll attempt to select a numeric field (for demonstration, look for 'MW' (molecular weight), 'Abundance', or any candidate in the columns).

In [ ]:
# Select a numeric field for analysis (change as per revealed columns)
candidates = ['MW', 'MolecularWeight', 'Abundance', 'NormalizedAbundance', 'Coverage', 'PeptideCounts', 'pI']
numeric_field = None
df = dataframes.get(main_record_set_id)
for col in df.columns:
    if any([c.lower() in col.lower() for c in candidates]):
        numeric_field = col
        break
if not numeric_field:
    # Default to first numeric column if possible
    for col in df.select_dtypes(include='number').columns:
        numeric_field = col
        break
assert numeric_field, 'No numeric field found for EDA.'
print(f"Using numeric field for demonstration: {numeric_field}")

# Try to find a grouping field (e.g., 'Description', 'Gene', 'Sample', etc.)
group_candidates = ['Description', 'Gene', 'Sample', 'Protein', 'Accession']
group_field = None
for col in df.columns:
    if any([c.lower() in col.lower() for c in group_candidates]):
        group_field = col
        break
if group_field:
    print(f"Grouping by field: {group_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouped analysis
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).to_frame()
    print(f"Top groups by mean {numeric_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field], kde=True, bins=30, color='steelblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping, visualize top groups
if group_field and group_field in df.columns:
    top_groups = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 4))
    sns.barplot(y=top_groups.index, x=top_groups.values, orient='h', palette='viridis')
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.title(f"Top 10 {group_field}s by mean {numeric_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded metadata and records from the Croissant dataset using the `mlcroissant` library.
- Available record sets and fields were inspected by their `@id`.
- Data for the main record set was loaded into a DataFrame, numeric columns explored and visualized.
- Demonstrated basic filtering, normalization, and grouping of numeric data.

Further analyses may include deeper feature engineering, statistical tests, or machine learning workflows.